In [1]:
"""
Project 01: Credit Risk Prediction
Step 02: Data Preprocessing & Cleaning
"""

'\nProject 01: Credit Risk Prediction\nStep 02: Data Preprocessing & Cleaning\n'

In [2]:
import warnings

In [3]:
warnings.filterwarnings("ignore")

In [4]:
import pandas as pd
import numpy as np
import matplotlib

In [5]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

In [6]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in globals() else (os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [7]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [8]:
print("=" * 60)
print("STEP 02: Data Preprocessing & Cleaning")
print("=" * 60)
start_time = time.time()

STEP 02: Data Preprocessing & Cleaning


In [9]:
# ── Load Data ──────────────────────────────────────────────
print("\nLoading application_train.csv ...")
df = pd.read_csv(os.path.join(RAW, "application_train.csv"))
print(f"Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing values before: {df.isnull().sum().sum():,}")
print(f"Categorical cols before: {df.select_dtypes(include='object').shape[1]}")


Loading application_train.csv ...
Raw shape: 23,895 rows × 122 columns
Missing values before: 709,988
Categorical cols before: 16


In [10]:
# ── Step 1: Handle Anomaly — DAYS_EMPLOYED = 365243 ──────
print("\n[1/4] Handling anomaly sentinel values ...")
anomaly_count = (df["DAYS_EMPLOYED"] == 365243).sum()
print(
    f"  DAYS_EMPLOYED = 365243 count: {anomaly_count:,} ({anomaly_count / len(df):.1%})"
)


[1/4] Handling anomaly sentinel values ...
  DAYS_EMPLOYED = 365243 count: 4,253 (17.8%)


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["DAYS_EMPLOYED"].plot.hist(bins=50, ax=axes[0], color="#FF6584", alpha=0.8)
axes[0].set_title(
    "DAYS_EMPLOYED — Before Fix\n(sentinel spike at 365,243)", fontweight="bold"
)
axes[0].set_xlabel("Days Employed")

Text(0.5, 0, 'Days Employed')

In [12]:
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["DAYS_EMPLOYED"].dropna().plot.hist(bins=50, ax=axes[1], color="#6C63FF", alpha=0.8)
axes[1].set_title("DAYS_EMPLOYED — After Fix\n(clean distribution)", fontweight="bold")
axes[1].set_xlabel("Days Employed")

Text(0.5, 0, 'Days Employed')

In [13]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "anomaly_fix.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: anomaly_fix.png")

  Saved: anomaly_fix.png


In [14]:
# ── Step 2: Drop High-Missing Columns (>50%) ────────────
print("[2/4] Dropping high-missing columns ...")
missing_rate = df.isnull().mean()
cols_to_drop = missing_rate[missing_rate > 0.5].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f"  Dropped {len(cols_to_drop)} columns (>50% missing)")

[2/4] Dropping high-missing columns ...
  Dropped 40 columns (>50% missing)


In [15]:
# ── Step 3: Impute Missing Values ───────────────────────
print("[3/4] Imputing missing values ...")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

[3/4] Imputing missing values ...


In [16]:
for col in tqdm(num_cols, desc="  Numerical impute"):
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

  Numerical impute: 100%|██████████| 68/68 [00:00<00:00, 1589.78it/s]


In [17]:
for col in tqdm(cat_cols, desc="  Categorical impute"):
    if df[col].isnull().any():
        df[col] = df[col].fillna("Missing")

  Categorical impute: 100%|██████████| 14/14 [00:00<00:00, 571.98it/s]


In [18]:
print(f"  Missing values after: {df.isnull().sum().sum()}")

  Missing values after: 0


In [19]:
# ── Step 4: Encode Categoricals ──────────────────────────
print("[4/4] Encoding categorical columns ...")
label_encoders = {}
for col in tqdm(cat_cols, desc="  Label encoding"):
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

[4/4] Encoding categorical columns ...


  Label encoding: 100%|██████████| 14/14 [00:00<00:00, 290.48it/s]


In [20]:
print(f"  Encoded {len(cat_cols)} categorical columns")

  Encoded 14 categorical columns


In [21]:
# ── Data types after preprocessing ──────────────────────
dtype_counts = df.dtypes.astype(str).value_counts()

In [22]:
fig, ax = plt.subplots(figsize=(8, 4))
dtype_counts.plot.bar(ax=ax, color=["#6C63FF", "#FF6584", "#43E97B"], alpha=0.85)
ax.set_title("Data Types Distribution After Preprocessing", fontweight="bold")
ax.set_xlabel("Data Type")
ax.set_ylabel("Column Count")
for i, v in enumerate(dtype_counts):
    ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "dtype_distribution.png"), dpi=150, bbox_inches="tight"
)
plt.close()

In [23]:
# ── Why Median Imputation? ──────────────────────────────
col = "AMT_INCOME_TOTAL"
data = df[col].dropna()

In [24]:
fig, ax = plt.subplots(figsize=(10, 5))
data.clip(0, 1.5e6).plot.hist(bins=60, ax=ax, color="#6C63FF", alpha=0.8)
ax.axvline(
    data.mean(),
    color="#FF6584",
    lw=2.5,
    linestyle="--",
    label=f"Mean: {data.mean():,.0f}",
)
ax.axvline(
    data.median(),
    color="#43E97B",
    lw=2.5,
    linestyle="-",
    label=f"Median: {data.median():,.0f}",
)
ax.set_title("Income Distribution — Mean vs Median for Imputation", fontweight="bold")
ax.set_xlabel("Annual Income")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "median_vs_mean.png"), dpi=150, bbox_inches="tight")
plt.close()

In [25]:
# ── Save Processed Data ──────────────────────────────────
output_path = os.path.join(PROCESSED, "preprocessed.csv")
df.to_csv(output_path, index=False)
print(f"\nSaved preprocessed data: {output_path}")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


Saved preprocessed data: /home/ashutosh/Desktop/NO-AI-USE/ml_projects/01_credit_risk_prediction/data/processed/preprocessed.csv
Final shape: 23,895 rows × 82 columns


In [26]:
# Save label encoders
with open(os.path.join(PROCESSED, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

In [27]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 02 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")


STEP 02 COMPLETE | Time: 15.9s
